In [4]:
import pandas as pd
df = pd.read_csv(r"C:\Users\sumit\Downloads\customer_shopping_behavior (1).csv")


In [7]:
#cleaning the data 
df.dropna(inplace=True)
df.drop_duplicates(inplace=True)
df.columns=df.columns.str.lower()
df.columns=df.columns.str.strip().str.replace(" ","_")

In [8]:
df.head(3)

,customer_id,age,gender,item_purchased,category,purchase_amount_(usd),location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,frequency_of_purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly


In [9]:
#check missing values
print(df.isnull().sum())

customer_id               0
age                       0
gender                    0
item_purchased            0
category                  0
purchase_amount_(usd)     0
location                  0
size                      0
color                     0
season                    0
review_rating             0
subscription_status       0
shipping_type             0
discount_applied          0
promo_code_used           0
previous_purchases        0
payment_method            0
frequency_of_purchases    0
dtype: int64


In [10]:
#filling missing review_rating with avg
df['review_rating']=df['review_rating'].fillna(df['review_rating'].mean())

In [11]:
#renaming the column names 
df.rename(columns={
    'purchase_amount_(usd)':'purchase_amount',
    'frequency_of_purchases' : 'purchase_frequency'
},inplace=True)

In [12]:
#coverting DataTypes
df['review_rating']=df['review_rating'].astype(float)
df['purchase_amount']=df['purchase_amount'].astype(int)

In [ ]:
#Creating columns
df['spending_levels']=df['purchase_amount'].apply(
    lambda x:'High' if x > 70 else 'Medium' if x > 40 else 'Low'
)
df['age_group']=pd.cut(df['age'],bins=[0,25,40,60,100],labels=['Young','Adult','Middle Age','Senior'])

In [16]:
df.head(5)

,customer_id,age,gender,item_purchased,category,purchase_amount,location,size,color,season,review_rating,subscription_status,shipping_type,discount_applied,promo_code_used,previous_purchases,payment_method,purchase_frequency,spending_levels,age_group
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly,Medium,Middle Age
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly,Medium,Young
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly,High,Middle Age
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly,High,Young
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually,Medium,Middle Age


In [17]:
#grouping 
df.groupby('category')['purchase_amount'].sum().sort_values(ascending=False)

category
Clothing       102968
Accessories     73480
Footwear        35778
Outerwear       18403
Name: purchase_amount, dtype: int64

In [21]:
df.groupby('gender')['purchase_amount'].sum()
#avg
df.groupby('gender')['purchase_amount'].mean()

gender
Female    60.249199
Male      59.440918
Name: purchase_amount, dtype: float64

In [22]:
df.groupby('age_group')['purchase_amount'].mean()

age_group
Young         60.567138
Adult         59.721207
Middle Age    59.452285
Senior        59.507692
Name: purchase_amount, dtype: float64

In [26]:
from sqlalchemy import create_engine

server = 'localhost\\SQLDEV'   
database = 'customer_db'

engine = create_engine(
    f"mssql+pyodbc://@{server}/{database}?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes",
    fast_executemany=True
)

In [28]:
conn=engine.connect()
print("connected")

connected


In [29]:
df.to_sql(
    name='customer_data',
    con=engine,
    if_exists='replace',
    index=False,
    chunksize=500  
)

-8